In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

root = Path.cwd()
if not (root / "src").is_dir():
    root = root.parent
sys.path.insert(0, str(root))

from src.labels import RUNS, find_onset, op_hours, run_table

df = pd.read_parquet(root / "data" / "processed_features.parquet")
tables = {f"{s}_{b}": run_table(df, s, b) for s, b in RUNS}

In [ ]:
u = tables["set1_b3"]
dt = u["timestamp"].diff().dt.total_seconds().div(60)      # minutes between snapshots

print(dt.round().value_counts().head(8))                    # expect a spike at 10, a few at 5
print("first 45 intervals:", dt.iloc[1:46].round().tolist())

big = dt[dt > 15]
print(f"\n{len(big)} gaps > 15 min, totalling {big.sum() / 60:.0f} h")
print(u.loc[big.index, ["idx", "timestamp"]]
        .assign(gap_min=big.round(), life=u.loc[big.index, "life"].round(2))
        .sort_values("gap_min", ascending=False).head(10))

In [ ]:
u = tables["set1_b3"]
dt = u["timestamp"].diff().dt.total_seconds().div(60)
tail = u.assign(gap_min=dt.round())
print(tail[(tail["life"] > 0.95) & (tail["gap_min"] > 15)][["idx", "timestamp", "gap_min", "life"]])

In [ ]:
g = df[(df["set"] == "set1") & (df["bearing"] == "b3")].groupby("idx")["rms"].mean()
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(g.index[:400], g.iloc[:400], lw=0.8)
for i in (156, 235): ax.axvline(i, color="r", ls=":")
ax.set_xlabel("snapshot"); ax.set_ylabel("RMS (B3)"); plt.show()

In [ ]:
def op_hours(u, max_step_min=15):
    dt = u["timestamp"].diff().dt.total_seconds().div(3600).fillna(0)
    return dt.clip(upper=max_step_min / 60).cumsum()      # any gap counts as at most 15 min

for name, u in tables.items():
    i = find_onset(u, 1.25, 20)
    op = op_hours(u)
    dt = u["timestamp"].diff().dt.total_seconds().div(60)
    print(f"{name}: calendar {u['rul_hours'].iloc[i]:.1f} h | operating {op.iloc[-1] - op.iloc[i]:.1f} h | "
          f"gaps >15 min after onset: {(dt.iloc[i:] > 15).sum()}")

In [ ]:
for name, u in tables.items():
    dt = u["timestamp"].diff().dt.total_seconds().div(60)
    print(name, dt.round().value_counts().head(3).to_dict(),
          "| gaps >15 min:", (dt > 15).sum(), f"({dt[dt > 15].sum() / 60:.0f} h)")